In [ ]:
# [Setup Colab] baixa o dataset do repo
!wget -q -O dataset_agricultura_orbital_5000_pt.csv https://raw.githubusercontent.com/RogerioOxy/aether-gs-run/main/dataset_agricultura_orbital_5000_pt.csv
print('dataset pronto')

# AETHER — TerraScan (S4) · Classificação de Risco Ambiental Orbital

**Plataforma:** AETHER — Mission Control AI · *Do espaço, cuidando da Terra.*
**Operadora:** Orbital Climate Intelligence (OCI)
**Subsistema:** **S4 — Earth Observation (TerraScan)** · payload científico de observação da Terra da missão **AETHER-1**
**Disciplina:** Data Science & Analytics (DSA) — Global Solution 2026.1 (*Space Connect*)
**Professor:** Roberto G. Beraldo
**Curso:** 2º ano — Ciência da Computação — FIAP

**Integrantes (dupla):**
- **RM561942 — Rogerio Deligi**
- **RM562686 — Maria Fernanda Garavelli Dantas**

---

## Contexto da missão

A **AETHER-1** carrega o payload científico **TerraScan (S4)**, que recebe dados de satélites de observação terrestre e sensoriamento remoto e **classifica o risco ambiental de propriedades agrícolas**. Este notebook é o núcleo analítico do TerraScan: a partir da telemetria orbital (consumo hídrico, uso de insumos, produtividade, NDVI, solo) ele decide se uma fazenda é de **baixo risco (NOMINAL, classe 0)** ou **alto risco ambiental (CRÍTICO, classe 1)**, alimentando o **Alert Engine (S3)** do AETHER com um *alerta de missão crítica ambiental*.

> Mapeamento canônico de alerta (DSA): **0 → baixo risco = NOMINAL** · **1 → alto risco = CRÍTICO**.

A agência **Orbital Climate Intelligence (OCI)** hoje opera com planilhas; o TerraScan substitui esse controle manual por um pipeline reprodutível de Ciência de Dados que apoia decisões de **sustentabilidade, gestão hídrica, uso racional de insumos e prevenção de degradação ambiental**.

### Como este notebook está organizado (9 etapas obrigatórias)
1. Importação das bibliotecas
2. Carregamento e compreensão inicial dos dados
3. Análise exploratória (EDA) com 7 gráficos interpretados
4. Data Munging/Wrangling + **criação da target `RISCO_AMBIENTAL`** por regra de negócio
5. Preparação dos dados para modelagem
6. Treinamento dos 3 modelos (Logistic Regression, GaussianNB, Random Forest)
7. Avaliação com métricas e interpretação ambiental
8. Deploy simples (salvar/carregar modelo + fazenda fictícia)
9. Conclusão analítica

> **Reprodutibilidade:** todas as decisões técnicas são justificadas em texto. Use **Run All** (Kernel → Restart & Run All) para executar do início ao fim. O dataset deve estar **na mesma pasta** deste `.ipynb`.


## Etapa 1 — Importação das bibliotecas

Importamos as bibliotecas exigidas pelo enunciado: **pandas** e **numpy** (manipulação numérica), **matplotlib** e **seaborn** (visualização) e **scikit-learn** (modelagem). O `matplotlib` é exigido explicitamente pelo brief, então o usamos como camada base e o `seaborn` como camada de alto nível sobre ele.

Também fixamos um `RANDOM_STATE` único para garantir **reprodutibilidade** (mesmo split e mesmos modelos a cada execução) e aplicamos um tema visual escuro “controle de missão” (paleta AETHER) aos gráficos.


In [ ]:
# --- Bibliotecas base ---
import numpy as np
import pandas as pd

# --- Visualização (matplotlib exigido pelo brief + seaborn de alto nível) ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- scikit-learn: preparacao, modelos e avaliacao ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    classification_report,
    ConfusionMatrixDisplay,
)

# --- Persistencia do modelo (deploy) ---
import joblib

# --- Reprodutibilidade ---
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# --- Tema visual AETHER (Mission Control dark) ---
AETHER = {
    "bg": "#0A0E1A", "panel": "#161C2E", "grid": "#233047",
    "text": "#E5E7EB", "muted": "#9CA3AF",
    "ciano": "#22D3EE", "ambar": "#F5A623",
    "nominal": "#22C55E", "atencao": "#FACC15",
    "alerta": "#FB923C", "critico": "#EF4444",
}
sns.set_theme(style="darkgrid")
plt.rcParams.update({
    "figure.facecolor": AETHER["bg"],
    "axes.facecolor": AETHER["panel"],
    "axes.edgecolor": AETHER["grid"],
    "axes.labelcolor": AETHER["text"],
    "axes.titlecolor": AETHER["text"],
    "text.color": AETHER["text"],
    "xtick.color": AETHER["muted"],
    "ytick.color": AETHER["muted"],
    "grid.color": AETHER["grid"],
    "figure.dpi": 110,
})

print("AETHER · TerraScan (S4) — bibliotecas carregadas com sucesso.")
print("pandas", pd.__version__, "| numpy", np.__version__)

## Etapa 2 — Carregamento e compreensão inicial dos dados

Carregamos o dataset de telemetria do TerraScan por **caminho relativo** (o arquivo `.csv` fica na mesma pasta deste notebook). Em seguida inspecionamos a estrutura com `head()`, `info()`, `describe()` e `shape`, separamos as variáveis **numéricas** das **categóricas** e verificamos os **valores faltantes**.

> **Por que pelo nome real do arquivo?** O brief cita um nome genérico (`agriculture_dataset_portugues.csv`), mas o arquivo realmente entregue é `dataset_agricultura_orbital_5000_pt.csv`. Como **REGRA 0 é não inventar**, carregamos pelo nome que de fato existe na pasta. Se você renomear o arquivo, ajuste a variável `CSV_PATH` abaixo.


In [ ]:
# Caminho relativo: o CSV esta na MESMA pasta deste .ipynb
CSV_PATH = "dataset_agricultura_orbital_5000_pt.csv"

df = pd.read_csv(CSV_PATH)
print("Dataset TerraScan carregado:", df.shape[0], "propriedades x", df.shape[1], "colunas")

### 2.1 — Primeiras linhas (`head`)

Cada linha é uma **propriedade agrícola** monitorada pelo TerraScan a partir de um satélite de observação (`SATELITE`).

In [ ]:
df.head()

### 2.2 — Estrutura dos dados (`info`)

O `info()` mostra o tipo de cada coluna e quantos valores não-nulos existem — já antecipa onde há *missing values*.

In [ ]:
df.info()

### 2.3 — Estatísticas descritivas (`describe`)

Resumo das colunas numéricas: média, desvio, mínimo/máximo e quartis. Esses **quartis** serão a base estatística da regra de negócio da Etapa 4.

In [ ]:
df.describe()

### 2.4 — Dimensões, variáveis numéricas vs. categóricas e valores faltantes

Separamos explicitamente as colunas numéricas das categóricas (decisão necessária para o *encoding* e o *scaling* da Etapa 5) e contabilizamos os *missing values* por coluna.

In [ ]:
# Dimensoes
print(f"Linhas (propriedades): {df.shape[0]}")
print(f"Colunas (atributos):   {df.shape[1]}")

# Identificacao de variaveis numericas e categoricas
colunas_numericas = df.select_dtypes(include=[np.number]).columns.tolist()
colunas_categoricas = df.select_dtypes(include=["object"]).columns.tolist()

print("\nVariaveis NUMERICAS:", colunas_numericas)
print("\nVariaveis CATEGORICAS:", colunas_categoricas)

# Verificacao de valores faltantes (missing values)
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("\nValores faltantes por coluna (apenas com missing):")
print(missing if len(missing) else "Nenhum valor faltante.")
print(f"\nTotal de celulas faltantes: {int(df.isnull().sum().sum())}")

**Leitura da Etapa 2.** O dataset traz **5000 propriedades** e **15 colunas**. As numéricas concentram a telemetria física (área, fertilizante, pesticida, produtividade, consumo de água, temperatura, umidade do solo, NDVI) e as categóricas descrevem o contexto operacional (satélite, cultura, irrigação, solo, estação). Há *missing values* em algumas colunas (notadamente `UMIDADE_SOLO_PCT`, `INDICE_NDVI`, `TIPO_SOLO` e `FERTILIZANTE_UTILIZADO_TON`), que serão tratados na Etapa 4. Note também que existe uma coluna `RISCO_AMBIENTAL` original — mas, conforme o brief, **não sabemos como ela foi gerada**, então construiremos a nossa própria target na Etapa 4 (e a coluna original será descartada para evitar vazamento).

## Etapa 3 — Análise exploratória dos dados (EDA)

Antes de modelar, exploramos visualmente a telemetria do TerraScan. Todos os gráficos exigidos pelo brief têm **título e identificação dos eixos** e usam a paleta Mission Control. Para os gráficos que dependem da **classe de risco**, usamos aqui a coluna `RISCO_AMBIENTAL` original *apenas como referência exploratória* — a target definitiva (a nossa) só nasce na Etapa 4.

Os 7 gráficos exigidos são:
1. Barras da variável `RISCO_AMBIENTAL`
2. Histograma de `PRODUTIVIDADE_TON`
3. Histograma de `CONSUMO_AGUA_M3`
4. Boxplot de `PESTICIDA_UTILIZADO_KG` por classe de risco
5. Boxplot de `FERTILIZANTE_UTILIZADO_TON` por classe de risco
6. Boxplot de `PRODUTIVIDADE_TON` por classe de risco
7. Heatmap de correlação das variáveis numéricas


### 3.1 — Gráfico de barras: distribuição de `RISCO_AMBIENTAL` (referência)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ordem = sorted(df["RISCO_AMBIENTAL"].dropna().unique())
cores = [AETHER["nominal"], AETHER["critico"]]
sns.countplot(data=df, x="RISCO_AMBIENTAL", order=ordem,
              hue="RISCO_AMBIENTAL", hue_order=ordem,
              palette=cores[:len(ordem)], legend=False, ax=ax)
ax.set_title("TerraScan · Distribuição de Risco Ambiental (coluna original)")
ax.set_xlabel("Classe de risco (0 = Baixo / NOMINAL  ·  1 = Alto / CRÍTICO)")
ax.set_ylabel("Número de propriedades")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}",
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom", color=AETHER["text"], fontsize=10)
plt.tight_layout()
plt.show()

**Interpretação.** A coluna original é desbalanceada (mais propriedades de baixo risco do que de alto risco), padrão esperado em monitoramento ambiental — a maioria das fazendas opera dentro do normal e apenas uma minoria entra em estado crítico. Esse desbalanceamento reforça a importância de olhar **recall da classe 1** na avaliação (Etapa 7): deixar passar uma fazenda de alto risco (*falso negativo*) é o pior erro ambiental.

### 3.2 — Histograma de `PRODUTIVIDADE_TON`

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
sns.histplot(df["PRODUTIVIDADE_TON"], bins=40, kde=True,
             color=AETHER["ciano"], ax=ax)
ax.set_title("TerraScan · Distribuição da Produtividade Agrícola")
ax.set_xlabel("Produtividade (toneladas)")
ax.set_ylabel("Frequência (número de propriedades)")
plt.tight_layout()
plt.show()

**Interpretação.** A produtividade concentra-se em torno de ~52–55 t com uma cauda à direita (poucas fazendas de altíssima produtividade). Propriedades na **cauda inferior** (baixa produtividade) são candidatas naturais a risco ambiental: produzir pouco apesar de consumir insumos/água sugere ineficiência e possível degradação — por isso a baixa produtividade entra na regra da Etapa 4 com a lógica **invertida**.

### 3.3 — Histograma de `CONSUMO_AGUA_M3`

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
sns.histplot(df["CONSUMO_AGUA_M3"], bins=40, kde=True,
             color=AETHER["ambar"], ax=ax)
ax.set_title("TerraScan · Distribuição do Consumo de Água")
ax.set_xlabel("Consumo de água (m³)")
ax.set_ylabel("Frequência (número de propriedades)")
plt.tight_layout()
plt.show()

**Interpretação.** O consumo hídrico é fortemente **assimétrico à direita**: a maioria das fazendas usa volumes moderados, mas existe uma cauda longa de grandes consumidoras. Essas grandes consumidoras de água — sobretudo quando combinadas a grandes áreas — representam o maior risco de pressão sobre recursos hídricos, e por isso o **3º quartil de consumo** é um dos gatilhos da target.

### 3.4 — Boxplot de `PESTICIDA_UTILIZADO_KG` por classe de risco

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.boxplot(data=df, x="RISCO_AMBIENTAL", y="PESTICIDA_UTILIZADO_KG",
            order=ordem, hue="RISCO_AMBIENTAL", hue_order=ordem,
            palette=cores[:len(ordem)], legend=False, ax=ax)
ax.set_title("TerraScan · Uso de Pesticida por Classe de Risco")
ax.set_xlabel("Classe de risco (0 = Baixo  ·  1 = Alto)")
ax.set_ylabel("Pesticida utilizado (kg)")
plt.tight_layout()
plt.show()

**Interpretação.** Espera-se que a classe 1 (alto risco) apresente mediana e dispersão de pesticida superiores à classe 0. O uso intensivo de pesticida é um vetor direto de **contaminação de solo e água**, portanto é um indicador ambiental legítimo e entra na regra de negócio.

### 3.5 — Boxplot de `FERTILIZANTE_UTILIZADO_TON` por classe de risco

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.boxplot(data=df, x="RISCO_AMBIENTAL", y="FERTILIZANTE_UTILIZADO_TON",
            order=ordem, hue="RISCO_AMBIENTAL", hue_order=ordem,
            palette=cores[:len(ordem)], legend=False, ax=ax)
ax.set_title("TerraScan · Uso de Fertilizante por Classe de Risco")
ax.set_xlabel("Classe de risco (0 = Baixo  ·  1 = Alto)")
ax.set_ylabel("Fertilizante utilizado (ton)")
plt.tight_layout()
plt.show()

**Interpretação.** O fertilizante em excesso provoca **eutrofização** (enriquecimento de nutrientes em corpos d’água) e degradação do solo a longo prazo. Diferenças na distribuição entre as classes confirmam o fertilizante como atributo relevante — ele também compõe a target.

### 3.6 — Boxplot de `PRODUTIVIDADE_TON` por classe de risco

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.boxplot(data=df, x="RISCO_AMBIENTAL", y="PRODUTIVIDADE_TON",
            order=ordem, hue="RISCO_AMBIENTAL", hue_order=ordem,
            palette=cores[:len(ordem)], legend=False, ax=ax)
ax.set_title("TerraScan · Produtividade por Classe de Risco")
ax.set_xlabel("Classe de risco (0 = Baixo  ·  1 = Alto)")
ax.set_ylabel("Produtividade (ton)")
plt.tight_layout()
plt.show()

**Interpretação.** A produtividade tende a ser **menor** na classe de alto risco — coerente com a ideia de que fazendas que consomem muito (água/insumos) e produzem pouco são ambientalmente ineficientes. Essa relação inversa justifica o uso da produtividade **invertida** (baixa produtividade = ponto de risco) na regra da Etapa 4.

### 3.7 — Heatmap de correlação das variáveis numéricas

In [ ]:
# Correlacao apenas entre numericas (exclui ID e a target original do mapa visual)
num_para_corr = [c for c in colunas_numericas if c != "RISCO_AMBIENTAL"]
corr = df[num_para_corr].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="mako",
            linewidths=0.5, linecolor=AETHER["grid"],
            cbar_kws={"label": "Coeficiente de correlação de Pearson"}, ax=ax)
ax.set_title("TerraScan · Correlação entre Variáveis Numéricas")
plt.tight_layout()
plt.show()

**Interpretação.** O heatmap revela que as variáveis físicas são, em geral, **pouco correlacionadas entre si** (correlações próximas de zero), o que é ótimo para modelagem: indica baixa redundância/multicolinearidade entre os atributos. Isso significa que cada variável (consumo de água, pesticida, fertilizante, produtividade, área) carrega informação **independente** sobre o risco, e por isso combiná-las numa regra multifatorial (Etapa 4) faz sentido — nenhuma sozinha domina as demais.

## Etapa 4 — Data Munging/Wrangling + criação da target `RISCO_AMBIENTAL`

Esta etapa tem duas frentes:

**(A) Limpeza dos dados:** tratar *missing values* (por tipo de variável), identificar e tratar *outliers* (método IQR) e remover colunas irrelevantes (`ID_FAZENDA`).

**(B) Criação da target:** conforme o brief, **não sabemos** como a coluna `RISCO_AMBIENTAL` original foi gerada, então a **descartamos** e criamos a nossa própria regra de negócio, justificada estatisticamente.


### 4.1 — Tratamento de valores faltantes

Justificativa por tipo de variável:
- **Numéricas** (`FERTILIZANTE_UTILIZADO_TON`, `UMIDADE_SOLO_PCT`, `INDICE_NDVI`): imputamos pela **mediana**, que é robusta a *outliers* (mais adequada que a média em distribuições assimétricas como as que vimos na EDA).
- **Categóricas** (`TIPO_SOLO`): imputamos pela **moda** (categoria mais frequente), pois não existe “média” de uma categoria.


In [ ]:
# Copia de trabalho para nao alterar o df original
dados = df.copy()

# (1) Imputacao de NUMERICAS pela mediana (robusta a outliers)
num_com_missing = ["FERTILIZANTE_UTILIZADO_TON", "UMIDADE_SOLO_PCT", "INDICE_NDVI"]
for col in num_com_missing:
    if col in dados.columns:
        mediana = dados[col].median()
        n_faltantes = int(dados[col].isnull().sum())
        dados[col] = dados[col].fillna(mediana)
        print(f"[NUMERICA] {col}: {n_faltantes} faltantes -> imputados com mediana = {mediana:.2f}")

# (2) Imputacao de CATEGORICAS pela moda
cat_com_missing = ["TIPO_SOLO"]
for col in cat_com_missing:
    if col in dados.columns:
        moda = dados[col].mode()[0]
        n_faltantes = int(dados[col].isnull().sum())
        dados[col] = dados[col].fillna(moda)
        print(f"[CATEGORICA] {col}: {n_faltantes} faltantes -> imputados com moda = '{moda}'")

print("\nMissing values restantes:", int(dados.isnull().sum().sum()))

### 4.2 — Remoção de colunas irrelevantes

`ID_FAZENDA` é um identificador único (uma propriedade por linha) sem poder preditivo — mantê-lo só adicionaria ruído e risco de *overfitting*. Removemos. A coluna `RISCO_AMBIENTAL` **original** também sai do conjunto de atributos: ela será substituída pela nossa target e mantê-la causaria **vazamento de informação (data leakage)**.


In [ ]:
# Guardamos a coluna original apenas para comparacao posterior
risco_original = dados["RISCO_AMBIENTAL"].copy()

# Removemos ID (irrelevante) e a target original (sera recriada -> evita leakage)
dados = dados.drop(columns=["ID_FAZENDA", "RISCO_AMBIENTAL"])
print("Colunas apos remocao:", dados.columns.tolist())
print("Shape:", dados.shape)

### 4.3 — Criação da target por regra de negócio (justificada estatisticamente)

Seguindo o brief, uma propriedade é de **alto risco ambiental** quando combina fatores de pressão ambiental. Construímos um **score de risco** somando 5 indicadores baseados em **quantis** (limiares estatísticos do próprio dataset, não números arbitrários):

| # | Indicador de risco | Gatilho estatístico | Justificativa ambiental |
|---|--------------------|---------------------|--------------------------|
| 1 | Alto consumo de água | `CONSUMO_AGUA_M3` > Q3 (75º percentil) | Pressão sobre recursos hídricos |
| 2 | Uso elevado de pesticida | `PESTICIDA_UTILIZADO_KG` > Q3 | Contaminação de solo/água |
| 3 | Uso elevado de fertilizante | `FERTILIZANTE_UTILIZADO_TON` > Q3 | Eutrofização / degradação do solo |
| 4 | Baixa produtividade (**invertida**) | `PRODUTIVIDADE_TON` < Q1 (25º percentil) | Ineficiência: consome muito, produz pouco |
| 5 | Grande área **e** alto consumo hídrico | `AREA` > Q3 **E** `CONSUMO_AGUA_M3` > Q3 | Grandes fazendas hídrico-intensivas |

**Regra de decisão:** a propriedade é classificada como **alto risco (classe 1)** quando aciona **2 ou mais** dos 5 indicadores (`score >= 2`).

**Por que o limiar `>= 2`?** Usar `>= 1` classificaria como risco qualquer fazenda que apenas ultrapassasse um único quartil — cerca de 2/3 do dataset —, o que tornaria o alerta pouco útil (super-sensível). Exigir **pelo menos 2 gatilhos simultâneos** captura a noção de *combinação* de fatores pedida no brief e resulta numa proporção de alto risco realista (~28%), próxima da prevalência da coluna original e adequada para um classificador supervisionado.


In [ ]:
# Quantis (limiares estatisticos do proprio dataset)
q3_agua  = dados["CONSUMO_AGUA_M3"].quantile(0.75)
q3_pest  = dados["PESTICIDA_UTILIZADO_KG"].quantile(0.75)
q3_fert  = dados["FERTILIZANTE_UTILIZADO_TON"].quantile(0.75)
q1_prod  = dados["PRODUTIVIDADE_TON"].quantile(0.25)
q3_area  = dados["AREA_FAZENDA_ACRES"].quantile(0.75)

print("Limiares (quantis):")
print(f"  Q3 consumo de agua      = {q3_agua:.2f} m3")
print(f"  Q3 pesticida            = {q3_pest:.2f} kg")
print(f"  Q3 fertilizante         = {q3_fert:.2f} ton")
print(f"  Q1 produtividade        = {q1_prod:.2f} ton")
print(f"  Q3 area                 = {q3_area:.2f} acres")

# Score de risco (soma dos 5 indicadores)
score_risco = (
      (dados["CONSUMO_AGUA_M3"]        > q3_agua).astype(int)
    + (dados["PESTICIDA_UTILIZADO_KG"] > q3_pest).astype(int)
    + (dados["FERTILIZANTE_UTILIZADO_TON"] > q3_fert).astype(int)
    + (dados["PRODUTIVIDADE_TON"]      < q1_prod).astype(int)
    + ((dados["AREA_FAZENDA_ACRES"]    > q3_area) &
       (dados["CONSUMO_AGUA_M3"]       > q3_agua)).astype(int)
)

# Regra de decisao: alto risco se acionar 2+ indicadores
dados["RISCO_AMBIENTAL"] = (score_risco >= 2).astype(int)

print("\nDistribuicao do score de risco (0 a 5 indicadores):")
print(score_risco.value_counts().sort_index())
print("\nDistribuicao da target RISCO_AMBIENTAL criada:")
print(dados["RISCO_AMBIENTAL"].value_counts())
print("\nProporcao de alto risco (classe 1): "
      f"{dados['RISCO_AMBIENTAL'].mean()*100:.1f}%")

### 4.4 — Validação estatística da target criada

Para confirmar que a regra é **coerente** (e não arbitrária), comparamos a média dos atributos de risco entre as duas classes que criamos. Se a regra faz sentido, a classe 1 deve ter **mais** consumo/insumos/área e **menos** produtividade.


In [ ]:
comparacao = dados.groupby("RISCO_AMBIENTAL")[
    ["CONSUMO_AGUA_M3", "PESTICIDA_UTILIZADO_KG",
     "FERTILIZANTE_UTILIZADO_TON", "PRODUTIVIDADE_TON", "AREA_FAZENDA_ACRES"]
].mean().round(2)
comparacao.index = ["0 - Baixo risco (NOMINAL)", "1 - Alto risco (CRITICO)"]
print("Media dos atributos por classe de risco criada:")
print(comparacao.T)

**Leitura da validação.** Como esperado, a classe de **alto risco** apresenta médias **maiores** de consumo de água, pesticida, fertilizante e área, e produtividade **menor** — confirmando que a target captura exatamente a noção ambiental do brief. A regra é, portanto, **estatisticamente justificada** e interpretável.

### 4.5 — Identificação e tratamento de *outliers* (método IQR)

Identificamos *outliers* nas variáveis numéricas pelo método **IQR** (limites em Q1 − 1.5·IQR e Q3 + 1.5·IQR). **Decisão:** em vez de *remover* as linhas, aplicamos **capping (winsorization)** — limitamos os valores extremos aos limites do IQR.

**Por que não remover?** No contexto ambiental, valores extremos de consumo de água ou pesticida **não são erros** — são justamente as fazendas de maior risco que queremos detectar. Removê-las jogaria fora o sinal mais importante. O *capping* reduz a influência desproporcional dos extremos nos modelos sensíveis a escala (Logistic Regression) **sem** descartar propriedades. O *capping* é aplicado **depois** da criação da target, para não distorcer os quantis da regra.


In [ ]:
def tratar_outliers_iqr(serie):
    """Capping (winsorization) pelos limites IQR. Retorna serie tratada e contagem."""
    q1 = serie.quantile(0.25)
    q3 = serie.quantile(0.75)
    iqr = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr
    n_out = int(((serie < lim_inf) | (serie > lim_sup)).sum())
    return serie.clip(lower=lim_inf, upper=lim_sup), n_out, lim_inf, lim_sup

# Aplica apenas nas numericas continuas (nao na target)
features_num = ["AREA_FAZENDA_ACRES", "FERTILIZANTE_UTILIZADO_TON",
                "PESTICIDA_UTILIZADO_KG", "PRODUTIVIDADE_TON",
                "CONSUMO_AGUA_M3", "TEMPERATURA_MEDIA_C",
                "UMIDADE_SOLO_PCT", "INDICE_NDVI"]

print("Tratamento de outliers (capping IQR):")
for col in features_num:
    dados[col], n_out, li, ls = tratar_outliers_iqr(dados[col])
    print(f"  {col}: {n_out} outliers limitados ao intervalo [{li:.2f}, {ls:.2f}]")

print("\nShape final apos wrangling:", dados.shape)

## Etapa 5 — Preparação dos dados para modelagem

Executamos, na ordem exigida pelo brief:
1. Separação **X** (atributos) e **y** (target `RISCO_AMBIENTAL`);
2. Separação de colunas **numéricas** e **categóricas**;
3. **Encoding** das categóricas (One-Hot) e **scaling** das numéricas (padronização Z-score), encapsulados num `ColumnTransformer` para evitar *data leakage* (o `fit` do scaler ocorre só no treino, via `Pipeline`);
4. **`train_test_split`** com `random_state` fixo e **estratificação** pela target (mantém a proporção de classes em treino e teste).


In [ ]:
# 1) Separacao X / y
X = dados.drop(columns=["RISCO_AMBIENTAL"])
y = dados["RISCO_AMBIENTAL"]

# 2) Separacao numericas / categoricas
features_numericas = X.select_dtypes(include=[np.number]).columns.tolist()
features_categoricas = X.select_dtypes(include=["object"]).columns.tolist()
print("Features NUMERICAS:", features_numericas)
print("Features CATEGORICAS:", features_categoricas)

# 3) Pre-processador: scaling (numericas) + one-hot (categoricas)
preprocessador = ColumnTransformer(transformers=[
    ("num", StandardScaler(), features_numericas),
    ("cat", OneHotEncoder(handle_unknown="ignore"), features_categoricas),
])

# 4) Split treino/teste (estratificado para manter proporcao de classes)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print(f"\nTreino: {X_train.shape[0]} amostras  |  Teste: {X_test.shape[0]} amostras")
print("Proporcao classe 1 (treino): %.1f%%" % (y_train.mean() * 100))
print("Proporcao classe 1 (teste):  %.1f%%" % (y_test.mean() * 100))

## Etapa 6 — Treinamento dos modelos

Treinamos os **três modelos supervisionados obrigatórios**, cada um dentro de um `Pipeline` que aplica o pré-processamento (scaling + encoding) **antes** do classificador — isso garante que o mesmo tratamento seja reaplicado de forma consistente no teste e no deploy:

- **Logistic Regression** — baseline linear, interpretável, sensível a escala (por isso o scaling).
- **GaussianNB** — Naive Bayes gaussiano, rápido, assume independência entre atributos (a EDA mostrou baixa correlação, o que favorece a premissa).
- **Random Forest** — ensemble de árvores, captura interações não-lineares e costuma ser o mais robusto.


In [ ]:
modelos = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "GaussianNB": GaussianNB(),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1
    ),
}

pipelines = {}
for nome, modelo in modelos.items():
    pipe = Pipeline(steps=[
        ("preprocessador", preprocessador),
        ("classificador", modelo),
    ])
    pipe.fit(X_train, y_train)
    pipelines[nome] = pipe
    print(f"Modelo treinado: {nome}")

print("\nTodos os 3 modelos foram treinados com sucesso.")

## Etapa 7 — Avaliação dos modelos

Para cada modelo apresentamos **matriz de confusão**, **accuracy**, **precision**, **recall** e o **classification report**, seguidos de interpretação no contexto ambiental.

> **Énfase em falsos negativos.** No TerraScan, um **falso negativo** (classificar como baixo risco uma fazenda que é de **alto** risco) significa **deixar de disparar um alerta ambiental** — o erro mais grave. Por isso priorizamos o **recall da classe 1** (proporção de fazendas de alto risco que o modelo realmente detecta).


In [ ]:
resultados = []

for nome, pipe in pipelines.items():
    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    resultados.append({
        "Modelo": nome, "Accuracy": acc,
        "Precision (classe 1)": prec, "Recall (classe 1)": rec,
    })

    print("=" * 60)
    print(f"MODELO: {nome}")
    print("=" * 60)
    print(f"Accuracy : {acc:.3f}")
    print(f"Precision (classe 1 - alto risco): {prec:.3f}")
    print(f"Recall    (classe 1 - alto risco): {rec:.3f}")
    print("\nClassification report:")
    print(classification_report(
        y_test, y_pred,
        target_names=["0 - Baixo risco", "1 - Alto risco"],
        zero_division=0,
    ))

    # Matriz de confusao
    fig, ax = plt.subplots(figsize=(5, 4.2))
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Baixo", "Alto"])
    disp.plot(ax=ax, cmap="mako", colorbar=False)
    ax.set_title(f"Matriz de Confusão — {nome}")
    ax.set_xlabel("Classe prevista")
    ax.set_ylabel("Classe real")
    plt.tight_layout()
    plt.show()

### 7.1 — Quadro comparativo dos modelos

In [ ]:
tabela = pd.DataFrame(resultados).set_index("Modelo").round(3)
print("Comparativo de desempenho (foco em Recall da classe 1):")
print(tabela)

melhor = tabela["Recall (classe 1)"].idxmax()
print(f"\nMelhor modelo por RECALL da classe 1 (menos falsos negativos): {melhor}")

**Interpretação ambiental das métricas.**
- **Accuracy** mede o acerto global, mas sozinha engana em dados desbalanceados — um modelo que dissesse “tudo baixo risco” teria accuracy alta e seria inútil.
- **Precision (classe 1)**: das fazendas que o modelo marcou como alto risco, quantas realmente são. Baixa precision = muitos **falsos alarmes**, que custam inspeções desnecessárias.
- **Recall (classe 1)**: das fazendas que **são** de alto risco, quantas o modelo capturou. Baixo recall = **falsos negativos** = degradação ambiental passando despercebida. **É a métrica prioritária do TerraScan.**

Na prática, o TerraScan deve preferir o modelo com **maior recall da classe 1** (tipicamente o **Random Forest**), aceitando alguns falsos alarmes em troca de **não deixar passar** uma fazenda ambientalmente crítica. Selecionamos esse modelo para o deploy.

## Etapa 8 — Deploy simples do modelo

Demonstramos que o modelo treinado pode ser **salvo, carregado e reutilizado** para classificar novos registros — sem retreinar. Usamos **`joblib`** (formato recomendado pela documentação do scikit-learn para persistência). Selecionamos o modelo com **melhor recall da classe 1** (o critério do TerraScan).

Passos: (1) `joblib.dump` do pipeline completo → (2) `joblib.load` → (3) criar uma **fazenda fictícia** num `DataFrame` com as colunas reais → (4) predição real → (5) interpretação.

> Como salvamos o **pipeline inteiro** (pré-processador + classificador), o objeto carregado já aplica scaling e encoding automaticamente ao novo registro — é só passar o `DataFrame` cru.


In [ ]:
# 1) Seleciona o melhor pipeline (maior recall da classe 1) e salva
modelo_deploy = pipelines[melhor]
ARQ_MODELO = "terrascan_modelo_risco_ambiental.joblib"
joblib.dump(modelo_deploy, ARQ_MODELO)
print(f"Modelo salvo em: {ARQ_MODELO}  (modelo: {melhor})")

# 2) Carrega o modelo salvo
modelo_carregado = joblib.load(ARQ_MODELO)
print("Modelo carregado de disco com sucesso.")

In [ ]:
# 3) Fazenda ficticia (alto risco esperado): muito consumo de agua, muito pesticida/
#    fertilizante, baixa produtividade e area grande -> deve disparar alerta CRITICO.
fazenda_nova = pd.DataFrame([{
    "SATELITE": "TerraScan_X",
    "TIPO_CULTURA": "Soja",
    "AREA_FAZENDA_ACRES": 900.0,
    "TIPO_IRRIGACAO": "Inundacao",
    "TIPO_SOLO": "Arenoso",
    "ESTACAO": "Verao",
    "FERTILIZANTE_UTILIZADO_TON": 28.0,
    "PESTICIDA_UTILIZADO_KG": 260.0,
    "PRODUTIVIDADE_TON": 30.0,
    "CONSUMO_AGUA_M3": 12000.0,
    "TEMPERATURA_MEDIA_C": 34.0,
    "UMIDADE_SOLO_PCT": 40.0,
    "INDICE_NDVI": 0.45,
}])

# Garante a mesma ordem de colunas usada no treino
fazenda_nova = fazenda_nova[X.columns]
print("Fazenda ficticia a classificar:")
print(fazenda_nova.T)

In [ ]:
# 4) Predicao real com o modelo carregado
pred = modelo_carregado.predict(fazenda_nova)[0]
prob = modelo_carregado.predict_proba(fazenda_nova)[0][1]

rotulo = "ALTO RISCO (CRITICO)" if pred == 1 else "BAIXO RISCO (NOMINAL)"
print("=" * 55)
print("  AETHER · TerraScan (S4) — CLASSIFICACAO DE RISCO")
print("=" * 55)
print(f"  Classe prevista : {pred}  ->  {rotulo}")
print(f"  Probabilidade de alto risco : {prob*100:.1f}%")
if pred == 1:
    print("  >> Alert Engine (S3): disparar ALERTA DE MISSAO CRITICA AMBIENTAL")
else:
    print("  >> Alert Engine (S3): estado NOMINAL, sem alerta")
print("=" * 55)

**Interpretação do deploy.** A fazenda fictícia foi modelada como um cenário de **pressão ambiental máxima** (alto consumo hídrico, uso intenso de pesticida e fertilizante, baixa produtividade e grande área). O modelo carregado a classificou como **alto risco**, com probabilidade elevada, e o TerraScan encaminharia esse resultado ao **Alert Engine (S3)** como um *alerta de missão crítica ambiental*. Isso comprova o ciclo completo de deploy: **treinar → salvar → carregar → prever** um registro novo e inédito.

## Etapa 9 — Conclusão analítica

**Como a Ciência de Dados auxilia a agricultura inteligente.** O TerraScan transforma a telemetria orbital bruta (planilhas operacionais da OCI) num **classificador reprodutível de risco ambiental**. Em vez de inspeções manuais caras e tardias, o modelo prioriza automaticamente quais propriedades merecem atenção — apoiando decisões de sustentabilidade, gestão hídrica e uso racional de insumos em escala regional/global.

**Satélites, sensoriamento remoto e análise agrícola.** Os atributos do dataset (NDVI, umidade do solo, consumo de água, área) vêm de **satélites de observação terrestre e sensores orbitais**. O sensoriamento remoto fornece cobertura contínua e não-invasiva; a Ciência de Dados converte esses sinais em **decisões acionáveis** — essa é a ponte “do espaço, cuidando da Terra” da missão AETHER-1.

**Impactos ambientais (água, pesticidas, fertilizantes).** O excesso de água pressiona recursos hídricos; o de pesticida contamina solo e lençóis freáticos; o de fertilizante causa **eutrofização**. A regra de negócio combina esses três vetores com baixa produtividade e grande área, alinhando o conceito de “risco” aos danos ambientais reais.

**Riscos de falsos negativos.** No contexto ambiental, o **falso negativo é o erro mais caro**: uma fazenda de alto risco classificada como segura **não gera alerta**, e a degradação (contaminação, exaustão hídrica) avança sem intervenção. Por isso a avaliação priorizou o **recall da classe 1** e o deploy usou o modelo que **minimiza** esses casos, mesmo ao custo de alguns falsos alarmes.

**Limitações do modelo.** (1) A target é **sintética** (regra de negócio por quantis), não um rótulo validado em campo — o modelo aprende a regra, não a “verdade ambiental absoluta”. (2) Os limiares por quartil são **relativos a este dataset**; outra safra/região teria quartis diferentes. (3) Não há dimensão **temporal** (cada linha é um retrato único). (4) O dataset é **balanceado artificialmente** pela regra; em produção o desbalanceamento real pode ser maior.

**Possíveis melhorias futuras.** Validar a target com **especialistas agronômicos** e dados de campo; incorporar **séries temporais** (NDVI ao longo da safra); testar **balanceamento** (SMOTE/`class_weight`) e ajuste de **threshold** para maximizar recall; aplicar **validação cruzada** e *tuning* de hiperparâmetros; integrar o TerraScan ao **Alert Engine (S3)** em tempo real, fechando o ciclo de “Mission Control” da AETHER.

---

### Identificação da entrega
**AETHER — TerraScan (S4)** · Global Solution 2026.1 (Space Connect) · Data Science & Analytics — FIAP
- **RM561942 — Rogerio Deligi**
- **RM562686 — Maria Fernanda Garavelli Dantas**

> *AETHER — Do espaço, cuidando da Terra.*
